In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip install git+https://github.com/amazon-science/chronos-forecasting.git
!pip install pandas torch matplotlib numpy pyarrow fastparquet peft

  Cloning https://github.com/amazon-science/chronos-forecasting.git to /tmp/pip-req-build-lvbd0k5d
  Running command git clone --filter=blob:none --quiet https://github.com/amazon-science/chronos-forecasting.git /tmp/pip-req-build-lvbd0k5d
  Resolved https://github.com/amazon-science/chronos-forecasting.git to commit f889ae66477b53f6beb130f5c7b13590b29a1035
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import torch.nn as nn
from transformers import AutoModel, AutoImageProcessor
from chronos.chronos2.model import Chronos2Model
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
import io
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model, TaskType
import os
from PIL import Image
from tqdm import tqdm


# MULTIMODAL CHRONOS

In [ ]:
class VisionProjector(nn.Module):
    def __init__(self, input_dim=384, output_dim=16, hidden_dim=128):
        """
        Projects high-dimensional visual features (from DinoV2) into
        low-dimensional 'synthetic covariates' for Chronos.
        """
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)

class MultimodalChronos(nn.Module):
    def __init__(
        self,
        chronos_model_name="amazon/chronos-2",
        vision_model_name="facebook/dinov2-small",
        covariate_dim=16,
        freeze_vision=True
    ):
        super().__init__()
        self.covariate_dim = covariate_dim

        # 1. The "Eye" (Vision Backbone)
        print(f"Loading Vision Backbone: {vision_model_name}")
        self.vision_backbone = AutoModel.from_pretrained(vision_model_name)
        if freeze_vision:
            for param in self.vision_backbone.parameters():
                param.requires_grad = False

        vision_dim = self.vision_backbone.config.hidden_size

        # 2. The "Translator" (Projector)
        self.projector = VisionProjector(input_dim=vision_dim, output_dim=covariate_dim)

        # 3. The "Brain" (Chronos 2)
        self.chronos = None # Will be set via load_chonos or passed in.

    def load_chronos(self, pretrained_model_path="amazon/chronos-2", device_map="cuda"):
        from chronos import BaseChronosPipeline
        pipeline = BaseChronosPipeline.from_pretrained(pretrained_model_path, device_map=device_map, torch_dtype=torch.bfloat16)
        self.chronos = pipeline.model
        return pipeline.tokenizer

    def forward(
        self,
        context_tensor,        # (Batch, Time) - The target time series history (PV)
        pixel_values,          # (Batch, Total_Time, C, H, W) - Images for History + Future
        group_ids=None,        # (Batch) - Not used directly, we generate internal IDs
        future_target=None     # (Batch, Pred_Len) - The target time series future (PV)
    ):
        """
        Forward pass fusing Vision + Time using Group Attention.
        Strategy: Treat the K projected visual features as K auxiliary time series
        in the same group as the target PV series.
        """
        batch_size, _, c, h, w = pixel_values.shape
        device = context_tensor.device

        # --- A. Process Images (The "Eye") ---
        # Flatten batch and time to pass through Dino (Batch*Total_Time, C, H, W)
        flat_images = pixel_values.reshape(-1, c, h, w)

        with torch.no_grad():
            vision_outputs = self.vision_backbone(pixel_values=flat_images)
            raw_embeddings = vision_outputs.last_hidden_state[:, 0, :] # (B*T, VisionDim)

        # --- B. Project to Covariates (The "Translator") ---
        # (B*T, VisionDim) -> (B*T, CovariateDim)
        # Note: We keep it flattened for the projector, then reshape
        visual_flat = self.projector(raw_embeddings)

        # Reshape to (Batch, Total_Time, CovariateDim)
        # Total_Time = ContextLen + PredLen (usually)
        visual_seq = visual_flat.reshape(batch_size, -1, self.covariate_dim)

        # Split Visual Sequence into Context and Future parts
        context_len = context_tensor.shape[1]

        # Visual Context: (Batch, ContextLen, CovariateDim)
        visual_context = visual_seq[:, :context_len, :]

        # Visual Future: (Batch, PredLen, CovariateDim)
        # Check if we have future part
        pred_len = 0
        if future_target is not None:
            pred_len = future_target.shape[1]
            visual_future = visual_seq[:, context_len : context_len+pred_len, :]
        else:
             # Inference mode without future target
             # Chronos 2 usually generates autoregressively or predicts future.
             # If we want to guide generation, we need to pass 'future_covariates' logic?
             # But here we are using Group Attention.
             # We should pass visual_future as "future_target" for the auxiliary series?
             # Yes, we provide the "ground truth" for the visual series (projected)
             # so the model can attend to it while predicting the PV series.
             pass

        # --- C. Expand Batch for Group Attention ---
        # We want to stack:
        # 1. PV Series (1 channel)
        # 2. Visual Series (16 channels)
        # Total 17 "series" per sample.

        # 1. Prepare PV Series
        # PV Context: (Batch, 1, ContextLen)
        pv_context_expanded = context_tensor.unsqueeze(1)

        # PV Future: (Batch, 1, PredLen)
        if future_target is not None:
            pv_future_expanded = future_target.unsqueeze(1)

        # 2. Prepare Visual Series
        # Visual Context: (Batch, 16, ContextLen) -> Transpose to (Batch, CovariateDim, ContextLen)
        visual_context_transposed = visual_context.permute(0, 2, 1)

        if future_target is not None:
            visual_future_transposed = visual_future.permute(0, 2, 1)

        # 3. Concatenate
        # Context: (Batch, 1+16, ContextLen)
        combined_context = torch.cat([pv_context_expanded, visual_context_transposed], dim=1)

        # Future: (Batch, 1+16, PredLen)
        if future_target is not None:
            combined_future = torch.cat([pv_future_expanded, visual_future_transposed], dim=1)

        # 4. Flatten Batch Dimension
        # New Batch Size = Batch * 17
        num_series = 1 + self.covariate_dim

        flat_context = combined_context.reshape(-1, context_len) # (B*17, ContextLen)

        flat_future_target = None
        if future_target is not None:
            flat_future_target = combined_future.reshape(-1, pred_len) # (B*17, PredLen)

        # 5. Generate Group IDs
        # All 17 series for ample 'b' should have group_id 'b'
        # (Batch, 17) -> flat -> (Batch*17)
        # We can just iterate 0..Batch-1 and repeat
        ids = torch.arange(batch_size, device=device).unsqueeze(1).repeat(1, num_series).flatten()

        # 6. Generate Loss Mask (future_target_mask)
        # We only want to compute loss on the PV series (index 0 of each block), not Visual series.
        # Mask shape: (B*17, PredLen)
        # 1 = Train, 0 = Ignore
        if future_target is not None:
            # Create mask for one sample: (17, PredLen)
            # First row True, others False
            sample_mask = torch.zeros((num_series, pred_len), dtype=torch.bool, device=device)
            sample_mask[0, :] = True # Only train on PV

            # Repeat for batch
            flat_mask = sample_mask.unsqueeze(0).repeat(batch_size, 1, 1).reshape(-1, pred_len)
        else:
            flat_mask = None

        # --- D. Forward Pass ---
        # Calculate num_output_patches needed
        # Default patch size for Chronos 2 Small/Base is 16.
        # Ideally we fetch this from self.chronos.config, but to avoid accessing deep internals if structure varies:
        output_patch_size = 16
        if hasattr(self.chronos, "config"):
             # Chronos2Config usually has patch_size or similar
             # Based on user logs: (output_patch_embedding): ResidualBlock(..., out_features=336)
             # but input patch size is different?
             # Chronos 2 patch size is fixed at 16?
             # Let's assume 16 or try to read it.
             # User error: found: 96 > 1 * 16. So patch size is definitely 16.
             pass

        num_output_patches = 1
        if future_target is not None:
             import math
             pred_len = future_target.shape[1]
             num_output_patches = math.ceil(pred_len / output_patch_size)

        outputs = self.chronos(
            context=flat_context,
            group_ids=ids,
            future_target=flat_future_target,
            future_target_mask=flat_mask,
            num_output_patches=num_output_patches
        )

        return outputs


# MULTIMODAL DATASET

In [ ]:


class MultimodalDataset(Dataset):
    def __init__(self,
                 df,
                 prediction_length=96,
                 context_length=512,  # Chronos input context + prediction
                 chronos_tokenizer=None,
                 image_processor=None,
                 mode="train"):
        """
        Args:
            df: DataFrame with 'timestamp', 'item_id', 'pv_value', 'image' (dict or bytes).
            mode: 'train' (returns context + target + images) or 'inference' (context only).
        """
        self.df = df
        self.prediction_length = prediction_length
        self.context_length = context_length
        self.tokenizer = chronos_tokenizer
        self.image_processor = image_processor
        self.mode = mode

        # Group by series_id to get ready-to-sample sequences
        self.series_groups = [group for _, group in df.groupby('item_id')]

        # We need to create valid samples.
        # For simplicity in this demo, we can just take the last window for each series (if small dataset)
        # OR implement a sliding window logic.
        # Given the user's script, let's implement a sliding window or just "one sample per series"
        # if the series are pre-split.
        # The user's script splits Train/Test by cutting the tail.
        # Let's assume the passed 'df' is a single continuous train set,
        # and we want to sample random windows from it during training.

        self.samples = []
        min_len = context_length + (prediction_length if mode == "train" else 0)

        # Pre-calculate valid indices for sliding window
        # (This might be memory intensive if huge, but fine for prototype)
        for group in self.series_groups:
            series_len = len(group)
            if series_len < min_len:
                continue

            # Stride of 1 might be too much data, let's do stride of 12 (1 hour)
            stride = 12
            for i in range(0, series_len - min_len + 1, stride):
                # Store (group_idx, start_idx)
                # Note: We store reference to group, not copy data
                self.samples.append((group, i))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        group, start_idx = self.samples[idx]

        # Define window
        total_len = self.context_length + self.prediction_length
        # But wait, Chronos usually takes a context and predicts 'prediction_length'.
        # For training, we usually feed a sequence of length L and mask the end?
        # Or we feed context (A) and target (B).
        # Chronos 2 uses masked training. But usually we fine-tune on 'next token prediction' style
        # or just passed the full sequence?
        # Let's follow the standard: Input = Context [0...T], Target = [T...T+P]
        # Actually, for Chronos 2 pipeline, `fit` takes "target" and handles slicing?
        # Since we are writing a custom loop, we need to be explicit.
        # We will provide a sequence of length (context_length) as 'context'
        # And the next (prediction_length) as 'future_target'.
        # And Images for ALL of it (or just context?).
        # User wants "Forecasting Signals", so likely covers the future too if available (weather forecast).
        # We will assume we have images for the full window.

        window_end = start_idx + self.context_length
        if self.mode == "train":
            prediction_end = window_end + self.prediction_length
        else:
            prediction_end = window_end # No prediction part needed for input

        # 1. Get Time Series Data
        # Assuming Data is aligned and contiguous because we sorted in prep
        full_window = group.iloc[start_idx:prediction_end]

        # Extract values
        pv_values = full_window['pv_value'].values.astype(np.float32)

        # 2. Get Images
        images = []
        for img_data in full_window['image']:
            try:
                if isinstance(img_data, dict) and 'bytes' in img_data:
                    image = Image.open(io.BytesIO(img_data['bytes'])).convert("RGB")
                elif isinstance(img_data, bytes):
                    image = Image.open(io.BytesIO(img_data)).convert("RGB")
                else:
                    # Black image fallback
                    image = Image.new('RGB', (224, 224))
                    print("fall back to black image")
            except:
                 image = Image.new('RGB', (224, 224))
            images.append(image)

        # 3. Process Images -> Tensor
        # (Seq_Len, 3, 224, 224) usually.
        # AutoImageProcessor returns dict with 'pixel_values'
        image_inputs = self.image_processor(images, return_tensors="pt")
        pixel_values = image_inputs['pixel_values'] # (Seq_Len, 3, H, W)

        # 4. Prepare Tensors
        # Context is the first 'context_length'
        # Target is the next 'prediction_length'

        context_tensor = torch.tensor(pv_values[:self.context_length])

        # For training, we need the images aligned with context AND future?
        # The user said "This tells Chronos: Here are 16 extra time series...".
        # If we want to guide the prediction, we need the future covariates (the projected images).
        # So we pass ALL pixel_values.

        item = {
            "context": context_tensor, # (Context_Len)
            "pixel_values": pixel_values, # (Total_Len, 3, H, W)
        }

        if self.mode == "train":
            future_target = torch.tensor(pv_values[self.context_length:])
            item["future_target"] = future_target # (Pred_Len)

        return item

def collate_fn(batch):
    """
    Custom collate to stack tensors.
    """
    # Simply stack keys
    keys = batch[0].keys()
    collated = {}
    for k in keys:
        collated[k] = torch.stack([b[k] for b in batch])
    return collated


# TRAIN MULTIMODAL

In [ ]:


# Import our custom modules
#from multimodal_chronos import MultimodalChronos
#from multimodal_dataset import MultimodalDataset, collate_fn

# --- Configuration ---
base_dir = "/content/drive/MyDrive/FM_project/dataset"
DATA_PATH = os.path.join(base_dir, "skippd_train_aligned_v13_with_time_features.parquet")
OUTPUT_DIR = "/content/miltimodal_ckp"
VISION_MODEL = "facebook/dinov2-small"
CHRONOS_MODEL = "amazon/chronos-2"
BATCH_SIZE = 1 # Reduced from 4 to 1 to save memory
GRADIENT_ACCUMULATION_STEPS = 4 # Simulate batch size of 4
LEARNING_RATE = 1e-4
NUM_EPOCHS = 1
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 96
COVARIATE_DIM = 16


In [ ]:
def load_data(path):
    print(f"Loading dataset from {path}...")
    df = pd.read_parquet(path)

    # Ensuring timestamp sorting etc.
    if 'time' in df.columns and 'timestamp' not in df.columns:
        df['timestamp'] = pd.to_datetime(df['time'])
        if df['timestamp'].dt.tz is not None:
             df['timestamp'] = df['timestamp'].dt.tz_localize(None)

    df = df.sort_values(['series_id', 'timestamp']).reset_index(drop=True)

    # Rename for consistency with Dataset class expectations
    df = df.rename(columns={"series_id": "item_id", "pv": "pv_value"})
    return df

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # 1. Prepare Data
    df = load_data(DATA_PATH)

    # Image Processor for Dino
    image_processor = AutoImageProcessor.from_pretrained(VISION_MODEL)

    # Dataset
    dataset = MultimodalDataset(
        df=df,
        prediction_length=PREDICTION_LENGTH,
        context_length=CONTEXT_LENGTH,
        image_processor=image_processor,
        mode="train"
    )

    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    print(f"Dataset created with {len(dataset)} samples.")

    # 2. Prepare Model
    model = MultimodalChronos(
        chronos_model_name=CHRONOS_MODEL,
        vision_model_name=VISION_MODEL,
        covariate_dim=COVARIATE_DIM,
        freeze_vision=True
    )

    # Load Chronos part correctly
    from chronos import BaseChronosPipeline
    print("Loading Chronos Pipeline to get model...")
    pipeline = BaseChronosPipeline.from_pretrained(CHRONOS_MODEL, device_map=device, torch_dtype=torch.bfloat16)
    model.chronos = pipeline.model

    # Explicitly cast Vision Backbone to bfloat16 to save memory (it's frozen anyway)
    print("Casting Vision Backbone to bfloat16 to save memory...")
    model.vision_backbone.to(dtype=torch.bfloat16)

    # Also cast the Projector to bfloat16 to match the incoming embeddings
    print("Casting Projector to bfloat16...")
    model.projector.to(dtype=torch.bfloat16)

    # Move entire fusion model to device
    model.to(device)

    # Apply LoRA to Chronos
    peft_config = LoraConfig(
        inference_mode=False,
        r=8,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=[
            "self_attention.q",
            "self_attention.v",
            "self_attention.k",
            "self_attention.o",
            "output_patch_embedding.output_layer",
        ],
    )
    # We wrap just the chronos component
    model.chronos = get_peft_model(model.chronos, peft_config)
    model.chronos.print_trainable_parameters()

    # 3. Optimizer
    trainable_params = [p for p in model.projector.parameters()] + \
                       [p for p in model.chronos.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE)

    # 4. Training Loop
    model.train()
    print("Starting Training...")

    for epoch in range(NUM_EPOCHS):
        total_loss = 0
        steps = 0
        optimizer.zero_grad()

        for i, batch in enumerate(dataloader):
            # Move data to device and cast to bfloat16
            context = batch["context"].to(device).bfloat16()
            pixel_values = batch["pixel_values"].to(device).bfloat16()
            future_target = batch["future_target"].to(device).bfloat16()

            # Forward
            outputs = model(
                context_tensor=context,
                pixel_values=pixel_values,
                future_target=future_target
            )

            # Loss calculation
            loss = outputs.loss if hasattr(outputs, "loss") else None

            if loss is None:
                print("Warning: Model didn't return loss. Check implementation.")
                break

            # Normalize loss for gradient accumulation
            loss = loss / GRADIENT_ACCUMULATION_STEPS
            loss.backward()

            if (i + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()

            # Scale back for logging
            current_loss = loss.item() * GRADIENT_ACCUMULATION_STEPS
            total_loss += current_loss
            steps += 1

            # Aggressive cleanup
            del context, pixel_values, future_target, outputs, loss

            if steps % 10 == 0:
                print(f"Epoch {epoch+1} | Step {steps} | Loss: {current_loss:.4f}")
                # Optional: empty cache if really tight
                # torch.cuda.empty_cache()

        print(f"Epoch {epoch+1} Complete. Avg Loss: {total_loss/steps:.4f}")

    # 5. Save
    print("Saving Projector and Adapter...")
    torch.save(model.projector.state_dict(), os.path.join(OUTPUT_DIR, "vision_projector.pth"))
    model.chronos.save_pretrained(os.path.join(OUTPUT_DIR, "chronos_lora_adapter"))
    print("Done!")

if __name__ == "__main__":
    main()


Using device: cuda
Loading dataset from /content/drive/MyDrive/FM_project/dataset/skippd_train_aligned_v13_with_time_features.parquet...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Dataset created with 1406 samples.
Loading Vision Backbone: facebook/dinov2-small
Loading Chronos Pipeline to get model...


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


Casting Vision Backbone to bfloat16 to save memory...
Casting Projector to bfloat16...
trainable params: 1,206,912 || all params: 120,684,576 || trainable%: 1.0001
Starting Training...
Epoch 1 | Step 10 | Loss: 0.0853
Epoch 1 | Step 20 | Loss: 0.0329
Epoch 1 | Step 30 | Loss: 0.0264
Epoch 1 | Step 40 | Loss: 0.0294
Epoch 1 | Step 50 | Loss: 0.0735
Epoch 1 | Step 60 | Loss: 0.0880
Epoch 1 | Step 70 | Loss: 0.0705
Epoch 1 | Step 80 | Loss: 0.1141
Epoch 1 | Step 90 | Loss: 0.0880
Epoch 1 | Step 100 | Loss: 0.0721
Epoch 1 | Step 110 | Loss: 0.1867
Epoch 1 | Step 120 | Loss: 0.0196
Epoch 1 | Step 130 | Loss: 0.0158
Epoch 1 | Step 140 | Loss: 0.1978
Epoch 1 | Step 150 | Loss: 0.0435
Epoch 1 | Step 160 | Loss: 0.0334
Epoch 1 | Step 170 | Loss: 0.0290
Epoch 1 | Step 180 | Loss: 0.0250
Epoch 1 | Step 190 | Loss: 0.0337
Epoch 1 | Step 200 | Loss: 0.0879
Epoch 1 | Step 210 | Loss: 0.2869
Epoch 1 | Step 220 | Loss: 0.1308
Epoch 1 | Step 230 | Loss: 0.0494
Epoch 1 | Step 240 | Loss: 0.1234
Epoch 1 

KeyboardInterrupt: 

In [ ]:
from numba import cuda

try:
    device = cuda.get_current_device()
    device.reset()
except:
    print("Could not reset device, you may need to restart the runtime.")

# LOAD EMBEDDING

In [5]:
# --- Configuration ---
BASE_DIR = "/content/drive/MyDrive/FM_project/dataset"
INPUT_PATH_EMB = os.path.join(BASE_DIR, "skippd_train_aligned_v13_with_time_features.parquet")
OUTPUT_PATH_EMB = os.path.join(BASE_DIR,"skippd_train_embeddings.parquet")
VISION_MODEL = "facebook/dinov2-small"
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def main():
    print(f"Loading dataset from {INPUT_PATH_EMB}...")
    df = pd.read_parquet(INPUT_PATH_EMB)

    print(f"Loading Vision Model: {VISION_MODEL} on {DEVICE}...")
    processor = AutoImageProcessor.from_pretrained(VISION_MODEL)
    model = AutoModel.from_pretrained(VISION_MODEL).to(DEVICE)
    model.eval() # Set to eval mode

    # Optional: cast to bfloat16 for speed if supported
    model.to(dtype=torch.bfloat16)

    embeddings_list = []

    # Buffer for batching
    batch_images = []

    print("Starting extraction...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        img_data = row['image']
        try:
            if isinstance(img_data, dict) and 'bytes' in img_data:
                image = Image.open(io.BytesIO(img_data['bytes'])).convert("RGB")
            elif isinstance(img_data, bytes):
                image = Image.open(io.BytesIO(img_data)).convert("RGB")
            else:
                image = Image.new('RGB', (224, 224))
        except:
             image = Image.new('RGB', (224, 224))

        batch_images.append(image)

        if len(batch_images) >= BATCH_SIZE or idx == len(df) - 1:
            # Process batch
            inputs = processor(images=batch_images, return_tensors="pt").to(DEVICE)
            # Cast inputs to bfloat16
            inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

            with torch.no_grad():
                outputs = model(**inputs)

            # Extract CLS token: (Batch, 384)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().float().numpy()

            # Append to list
            embeddings_list.extend([emb for emb in batch_embeddings])

            batch_images = []

    # Add to DataFrame
    print(f"Extracted {len(embeddings_list)} embeddings.")

    # Verify alignment
    if len(embeddings_list) != len(df):
        print(f"Warning: Embeddings count {len(embeddings_list)} != DataFrame length {len(df)}")

    # Create new DataFrame with embeddings instead of raw images
    # We drop the heavy 'image' column and add 'visual_embedding'
    df_new = df.drop(columns=['image'])

    # Store as list of floats (easy to save in Parquet)
    df_new['visual_embedding'] = list(embeddings_list)

    print(f"Saving to {OUTPUT_PATH_EMB}...")
    df_new.to_parquet(OUTPUT_PATH_EMB)
    print("Done! 🎉")

if __name__ == "__main__":
    main()


Loading dataset from /content/drive/MyDrive/FM_project/dataset/skippd_train_aligned_v13_with_time_features.parquet...
Loading Vision Model: facebook/dinov2-small on cuda...
Starting extraction...


100%|██████████| 18667/18667 [02:44<00:00, 113.39it/s]


Extracted 18667 embeddings.
Saving to /content/drive/MyDrive/FM_project/dataset/skippd_train_embeddings.parquet...
Done! 🎉


In [3]:
class VisionProjector(nn.Module):
    def __init__(self, input_dim=384, output_dim=16, hidden_dim=128):
        """
        Projects high-dimensional visual features (from DinoV2) into
        low-dimensional 'synthetic covariates' for Chronos.
        """
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)

class MultimodalChronos(nn.Module):
    def __init__(
        self,
        chronos_model_name="amazon/chronos-2",
        vision_model_name="facebook/dinov2-small",
        covariate_dim=16,
        freeze_vision=True,
        use_precomputed_embeddings=True # New flag
    ):
        super().__init__()
        self.covariate_dim = covariate_dim
        self.use_precomputed = use_precomputed_embeddings

        # 1. The "Eye" (Vision Backbone)
        if not self.use_precomputed:
            print(f"Loading Vision Backbone: {vision_model_name}")
            self.vision_backbone = AutoModel.from_pretrained(vision_model_name)
            if freeze_vision:
                for param in self.vision_backbone.parameters():
                    param.requires_grad = False
            vision_dim = self.vision_backbone.config.hidden_size
        else:
            print("Using precomputed embeddings. Skipping Vision Backbone load.")
            self.vision_backbone = None
            # Assume DinoV2 Small dim if precomputed
            vision_dim = 384

        # 2. The "Translator" (Projector)
        self.projector = VisionProjector(input_dim=vision_dim, output_dim=covariate_dim)

        # 3. The "Brain" (Chronos 2)
        self.chronos = None

    def forward(
        self,
        context_tensor,        # (Batch, Time)
        pixel_values,          # (Batch, Total_Time, C, H, W) OR (Batch, Total_Time, Emb_Dim) if precomputed
        group_ids=None,
        future_target=None
    ):
        batch_size = pixel_values.shape[0]
        device = context_tensor.device

        # --- A. Process Images (The "Eye") ---
        if self.use_precomputed:
            # Input is already embeddings: (Batch, Total_Time, 384)
            # Just flatten relevant dims
            raw_embeddings = pixel_values.reshape(-1, pixel_values.shape[-1]) # (B*T, 384)
        else:
            # Full Vision Path
            _, _, c, h, w = pixel_values.shape
            flat_images = pixel_values.reshape(-1, c, h, w)
            with torch.no_grad():
                vision_outputs = self.vision_backbone(pixel_values=flat_images)
                raw_embeddings = vision_outputs.last_hidden_state[:, 0, :]

        # --- B. Project to Covariates (The "Translator") ---
        visual_flat = self.projector(raw_embeddings)

        # Reshape to (Batch, Total_Time, CovariateDim)
        visual_seq = visual_flat.reshape(batch_size, -1, self.covariate_dim)

        # Split Visual Sequence into Context and Future parts
        context_len = context_tensor.shape[1]

        visual_context = visual_seq[:, :context_len, :]

        pred_len = 0
        if future_target is not None:
            pred_len = future_target.shape[1]
            visual_future = visual_seq[:, context_len : context_len+pred_len, :]
        else:
             pass

        # --- C. Expand Batch for Group Attention ---
        pv_context_expanded = context_tensor.unsqueeze(1)

        if future_target is not None:
            pv_future_expanded = future_target.unsqueeze(1)

        visual_context_transposed = visual_context.permute(0, 2, 1)

        if future_target is not None:
            visual_future_transposed = visual_future.permute(0, 2, 1)

        combined_context = torch.cat([pv_context_expanded, visual_context_transposed], dim=1)

        if future_target is not None:
            combined_future = torch.cat([pv_future_expanded, visual_future_transposed], dim=1)

        num_series = 1 + self.covariate_dim

        flat_context = combined_context.reshape(-1, context_len)

        flat_future_target = None
        if future_target is not None:
            flat_future_target = combined_future.reshape(-1, pred_len)

        ids = torch.arange(batch_size, device=device).unsqueeze(1).repeat(1, num_series).flatten()

        if future_target is not None:
            sample_mask = torch.zeros((num_series, pred_len), dtype=torch.bool, device=device)
            sample_mask[0, :] = True
            flat_mask = sample_mask.unsqueeze(0).repeat(batch_size, 1, 1).reshape(-1, pred_len)
        else:
            flat_mask = None

        # --- D. Forward Pass ---
        output_patch_size = 16
        num_output_patches = 1
        if future_target is not None:
             import math
             pred_len = future_target.shape[1]
             num_output_patches = math.ceil(pred_len / output_patch_size)

        outputs = self.chronos(
            context=flat_context,
            group_ids=ids,
            future_target=flat_future_target,
            future_target_mask=flat_mask,
            num_output_patches=num_output_patches
        )

        return outputs


In [4]:
class MultimodalDataset(Dataset):
    def __init__(self, df, prediction_length, context_length, image_processor=None, mode="train", use_precomputed=False, stride=1):
        self.df = df
        self.prediction_length = prediction_length
        self.context_length = context_length
        self.image_processor = image_processor
        self.mode = mode
        self.use_precomputed = use_precomputed
        self.stride = stride

        # --- Optimization: Convert Dataframes to Tensors once ---
        # We need to map (item_id, local_index) -> global_index to slice correctly
        # Or even simpler: we just iterate the series, extract their data, and store a list of Tensors (one per series).

        self.series_data = {} # item_id -> { "pv": Tensor, "emb": Tensor }

        # Group by series_id
        grouped = self.df.groupby('item_id')

        # Create samples index
        self.samples = []

        print("Preprocessing dataset into tensors for speed...")
        for s_id, group in grouped:
            # Extract PV values as Float Tensor
            pv_values = torch.tensor(group['pv_value'].values, dtype=torch.float32)

            # Extract Embeddings
            embeddings = None
            images_list = None

            if use_precomputed:
                # Fast track: Stack all embeddings for this series at once
                # This is the slow operation, but we do it ONLY ONCE here.
                emb_list = group['visual_embedding'].values
                # Assuming emb_list is array of lists/arrays.
                # np.stack might still be slow if len is huge, but it's done once.
                # Note: np.vstack might be safer if shapes vary, but they shouldn't.
                embeddings = torch.tensor(np.stack(emb_list), dtype=torch.float32)

            else:
                 images_list = group['image'].values # Keep as list of bytes/dicts

            # Store in cached dict
            self.series_data[s_id] = {
                "pv": pv_values,
                "emb": embeddings,
                "img": images_list
            }

            series_len = len(group)

            # Generate Sample Indices
            if mode == "train":
                if series_len > context_length + prediction_length:
                    for i in range(0, series_len - context_length - prediction_length + 1, self.stride):
                         self.samples.append((s_id, i))
            else:
                if series_len >= context_length:
                    self.samples.append((s_id, series_len - context_length))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s_id, start_idx = self.samples[idx]

        series = self.series_data[s_id]

        # Slice ranges
        total_len = self.context_length + (self.prediction_length if self.mode == "train" else 0)
        end_idx = start_idx + total_len

        # 1. PV Values (Instant Tensor Slice)
        pv_slice = series["pv"][start_idx : end_idx]

        context = pv_slice[:self.context_length]
        future_target = pv_slice[self.context_length:] if self.mode=="train" else None

        if self.use_precomputed:
             # Instant Tensor Slice
             pixel_values = series["emb"][start_idx : end_idx]
        else:
            # Load Images (Legacy Slow Path)
            # We access the raw list of bytes/dicts stored in 'img'
            images_raw = series["img"][start_idx : end_idx]
            images = []

            for img_data in images_raw:
                try:
                    if isinstance(img_data, dict) and 'bytes' in img_data:
                        image = Image.open(io.BytesIO(img_data['bytes'])).convert("RGB")
                    elif isinstance(img_data, bytes):
                        image = Image.open(io.BytesIO(img_data)).convert("RGB")
                    else:
                        image = Image.new('RGB', (224, 224))
                except:
                     image = Image.new('RGB', (224, 224))
                images.append(image)

            # Processor
            if self.image_processor:
                encoding = self.image_processor(images, return_tensors="pt")
                pixel_values = encoding.pixel_values
            else:
                pixel_values = torch.zeros((total_len, 3, 224, 224))

        return {
            "context": context,
            "future_target": future_target,
            "pixel_values": pixel_values
        }

def collate_fn(batch):
    # Custom collate because items are Tensors of different lengths (if we had varying lengths),
    # but here they are fixed.
    # Just stack them.

    contexts = torch.stack([item['context'] for item in batch])
    pixel_values = torch.stack([item['pixel_values'] for item in batch])

    if batch[0]['future_target'] is not None:
        future_targets = torch.stack([item['future_target'] for item in batch])
    else:
        future_targets = None

    return {
        "context": contexts,
        "pixel_values": pixel_values,
        "future_target": future_targets
    }


In [5]:
# --- Configuration ---
#DATA_PATH = r"c:\Users\loren\Desktop\D vecchio\UNIVERSITA\MAGISTRALE\SecondYear\FoundationModel\FM_test\project_features\skippd_train_embeddings.parquet"
BASE_DIR = "/content/drive/MyDrive/FM_project/dataset"
DATA_PATH = os.path.join(BASE_DIR,"skippd_train_embeddings.parquet")
OUTPUT_DIR = "/content/multimodal_checkpoints_dim_16_batch_16"
VISION_MODEL = "facebook/dinov2-small"
CHRONOS_MODEL = "amazon/chronos-2"
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 1e-4
NUM_EPOCHS = 10
CONTEXT_LENGTH = 192 # Reduced to ~4 days for efficiency
PREDICTION_LENGTH = 96
COVARIATE_DIM = 16
STRIDE = 4 # 12 hours between samples (48 steps/day)

In [6]:
def load_data(path):
    print(f"Loading dataset from {path}...")
    df = pd.read_parquet(path)

    # Ensuring timestamp sorting etc.
    if 'time' in df.columns and 'timestamp' not in df.columns:
        df['timestamp'] = pd.to_datetime(df['time'])
        if df['timestamp'].dt.tz is not None:
             df['timestamp'] = df['timestamp'].dt.tz_localize(None)

    # Rename for consistency with Dataset class expectations
    # Check if we need to rename (precomputed file might have 'series_id' and 'pv')
    if "series_id" in df.columns:
        df = df.rename(columns={"series_id": "item_id", "pv": "pv_value"})

    df = df.sort_values(['item_id', 'timestamp']).reset_index(drop=True)
    return df

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # 1. Prepare Data
    df = load_data(DATA_PATH)

    # Dataset
    # image_processor is not needed if using precomputed embeddings
    dataset = MultimodalDataset(
        df=df,
        prediction_length=PREDICTION_LENGTH,
        context_length=CONTEXT_LENGTH,
        image_processor=None,
        mode="train",
        use_precomputed=True,
        stride=STRIDE
    )

    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
    print(f"Dataset created with {len(dataset)} samples.")

    # 2. Prepare Model
    model = MultimodalChronos(
        chronos_model_name=CHRONOS_MODEL,
        vision_model_name=VISION_MODEL,
        covariate_dim=COVARIATE_DIM,
        freeze_vision=True,
        use_precomputed_embeddings=True
    )

    # Load Chronos part correctly
    from chronos import BaseChronosPipeline
    print("Loading Chronos Pipeline to get model...")
    pipeline = BaseChronosPipeline.from_pretrained(CHRONOS_MODEL, device_map=device, torch_dtype=torch.bfloat16)
    model.chronos = pipeline.model

    # Explicitly cast Vision Backbone to bfloat16 to save memory (it's frozen anyway)
    # print("Casting Vision Backbone to bfloat16 to save memory...")
    # if model.vision_backbone is not None:
    #     model.vision_backbone.to(dtype=torch.bfloat16)

    # Also cast the Projector to bfloat16 to match the incoming embeddings
    print("Casting Projector to bfloat16...")
    model.projector.to(dtype=torch.bfloat16)

    # Move entire fusion model to device
    model.to(device)

    # Apply LoRA to Chronos
    peft_config = LoraConfig(
        inference_mode=False,
        r=8,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=[
            "self_attention.q",
            "self_attention.v",
            "self_attention.k",
            "self_attention.o",
            "output_patch_embedding.output_layer",
        ],
    )
    # We wrap just the chronos component
    model.chronos = get_peft_model(model.chronos, peft_config)
    model.chronos.print_trainable_parameters()

    # 3. Optimizer
    trainable_params = [p for p in model.projector.parameters()] + \
                       [p for p in model.chronos.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE)

    # 4. Training Loop
    model.train()
    print("Starting Training...")

    for epoch in range(NUM_EPOCHS):
        total_loss = 0
        steps = 0
        optimizer.zero_grad()

        for i, batch in enumerate(dataloader):
            # Move data to device and cast to bfloat16
            context = batch["context"].to(device).bfloat16()
            pixel_values = batch["pixel_values"].to(device).bfloat16()
            future_target = batch["future_target"].to(device).bfloat16()

            # Forward
            outputs = model(
                context_tensor=context,
                pixel_values=pixel_values,
                future_target=future_target
            )

            # Loss calculation
            loss = outputs.loss if hasattr(outputs, "loss") else None

            if loss is None:
                print("Warning: Model didn't return loss. Check implementation.")
                break

            # Normalize loss for gradient accumulation
            loss = loss / GRADIENT_ACCUMULATION_STEPS
            loss.backward()

            if (i + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()

            # Scale back for logging
            current_loss = loss.item() * GRADIENT_ACCUMULATION_STEPS
            total_loss += current_loss
            steps += 1

            # Aggressive cleanup
            del context, pixel_values, future_target, outputs, loss

            if steps % 10 == 0:
                print(f"Epoch {epoch+1} | Step {steps} | Loss: {current_loss:.4f}")
                # Optional: empty cache if really tight
                # torch.cuda.empty_cache()

        print(f"Epoch {epoch+1} Complete. Avg Loss: {total_loss/steps:.4f}")

    # 5. Save
    print("Saving Projector and Adapter...")
    torch.save(model.projector.state_dict(), os.path.join(OUTPUT_DIR, "vision_projector.pth"))
    model.chronos.save_pretrained(os.path.join(OUTPUT_DIR, "chronos_lora_adapter"))
    print("Done!")

if __name__ == "__main__":
    main()


Using device: cuda
Loading dataset from /content/drive/MyDrive/FM_project/dataset/skippd_train_embeddings.parquet...
Preprocessing dataset into tensors for speed...
Dataset created with 4453 samples.
Using precomputed embeddings. Skipping Vision Backbone load.
Loading Chronos Pipeline to get model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


Casting Projector to bfloat16...
trainable params: 1,206,912 || all params: 120,684,576 || trainable%: 1.0001
Starting Training...
Epoch 1 | Step 10 | Loss: 0.0800
Epoch 1 | Step 20 | Loss: 0.1046
Epoch 1 | Step 30 | Loss: 0.0525
Epoch 1 | Step 40 | Loss: 0.0801
Epoch 1 | Step 50 | Loss: 0.1048
Epoch 1 | Step 60 | Loss: 0.0793
Epoch 1 | Step 70 | Loss: 0.0870
Epoch 1 | Step 80 | Loss: 0.0943
Epoch 1 | Step 90 | Loss: 0.1137
Epoch 1 | Step 100 | Loss: 0.0520
Epoch 1 | Step 110 | Loss: 0.0760
Epoch 1 | Step 120 | Loss: 0.0953
Epoch 1 | Step 130 | Loss: 0.0713
Epoch 1 | Step 140 | Loss: 0.1071
Epoch 1 | Step 150 | Loss: 0.0952
Epoch 1 | Step 160 | Loss: 0.0642
Epoch 1 | Step 170 | Loss: 0.0611
Epoch 1 | Step 180 | Loss: 0.0550
Epoch 1 | Step 190 | Loss: 0.0969
Epoch 1 | Step 200 | Loss: 0.0455
Epoch 1 | Step 210 | Loss: 0.0888
Epoch 1 | Step 220 | Loss: 0.0834
Epoch 1 | Step 230 | Loss: 0.0797
Epoch 1 | Step 240 | Loss: 0.0728
Epoch 1 | Step 250 | Loss: 0.0776
Epoch 1 | Step 260 | Loss: 0

In [9]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from transformers import AutoImageProcessor, AutoModel
from chronos import BaseChronosPipeline
from peft import PeftModel, LoraConfig

In [12]:
# --- Configuration ---
BASE_DIR = "/content/drive/MyDrive/FM_project/dataset"
DATA_PATH = os.path.join(BASE_DIR,"skippd_train_embeddings.parquet")
CHECKPOINT_DIR = "/content/multimodal_checkpoints_dim_16_batch_16"
VISION_MODEL = "facebook/dinov2-small"
CHRONOS_MODEL = "amazon/chronos-2"
CONTEXT_LENGTH = 192
PREDICTION_LENGTH = 96
COVARIATE_DIM = 16
STRIDE = 96
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [19]:
# --- 1. Model Loading ---
def load_multimodal_model():
    print("Initializing Multimodal Architecture...")
    # Initialize wrapper just to hold the Projector structure
    model = MultimodalChronos(
        chronos_model_name=CHRONOS_MODEL,
        vision_model_name=VISION_MODEL,
        covariate_dim=COVARIATE_DIM,
        freeze_vision=True,
        use_precomputed_embeddings=True
    )

    # Load Base Chronos Pipeline
    print("Loading Base Chronos Pipeline...")
    pipeline = BaseChronosPipeline.from_pretrained(CHRONOS_MODEL, device_map=DEVICE, torch_dtype=torch.bfloat16)

    # Load LoRA Adapter onto the pipeline's internal model
    adapter_path = os.path.join(CHECKPOINT_DIR, "chronos_lora_adapter")
    if os.path.exists(adapter_path):
        print(f"Loading LoRA Adapter from {adapter_path}...")
        pipeline.model = PeftModel.from_pretrained(pipeline.model, adapter_path)
    else:
        print("Warning: LoRA adapter not found. Using base weights.")

    # Load Projector Weights
    projector_path = os.path.join(CHECKPOINT_DIR, "vision_projector.pth")
    if os.path.exists(projector_path):
        print(f"Loading Vision Projector from {projector_path}...")
        state_dict = torch.load(projector_path)
        model.projector.load_state_dict(state_dict)
    else:
        print("Warning: Vision Projector weights not found!")

    # Cast Projector and Move
    model.projector.to(dtype=torch.bfloat16)
    model.to(DEVICE)
    model.eval()

    return model, pipeline

# --- 2. Inference Function ---
@torch.no_grad()
def generate_multimodal_predictions(wrapper_model, pipeline, batch, num_samples=20):
    """
    Uses the pipeline.predict() method with manually fused multimodal context.
    """
    context = batch["context"].to(DEVICE).bfloat16() # (Batch, Context_Len)
    pixel_values = batch["pixel_values"].to(DEVICE).bfloat16()

    batch_size = context.shape[0]

    # --- A. Project Visuals ---
    raw_embeddings = pixel_values.reshape(-1, pixel_values.shape[-1])
    visual_flat = wrapper_model.projector(raw_embeddings)
    visual_seq = visual_flat.reshape(batch_size, -1, wrapper_model.covariate_dim)

    # Visual Context (aligned with PV context)
    visual_context = visual_seq[:, :CONTEXT_LENGTH, :]

    # --- B. Fuse Inputs (Group Input) ---
    pv_context_expanded = context.unsqueeze(1) # (B, 1, T)
    visual_context_transposed = visual_context.permute(0, 2, 1) # (B, C, T)

    combined_context = torch.cat([pv_context_expanded, visual_context_transposed], dim=1) # (B, 1+C, T)
    flat_context = combined_context.reshape(-1, CONTEXT_LENGTH) # (B*(1+C), T)

    # --- C. Group IDs ---
    num_series = 1 + wrapper_model.covariate_dim
    ids = torch.arange(batch_size, device=DEVICE).unsqueeze(1).repeat(1, num_series).flatten()

    # --- D. Pipeline Predict ---
    # Fix: pipeline.predict(context, ...) expects positional context (named 'inputs' or 'context')
    # Use positional argument to be safe.

    forecasts = pipeline.predict(
        flat_context,
        group_ids=ids,
        prediction_length=PREDICTION_LENGTH,
        num_samples=num_samples,
        limit_prediction_length=False
    )

    # Forecasts is an iterator or list of Forecast objects (one per flattened sequence)
    # We need to extract only the TARGET series (every (1+C)-th element)

    target_indices = np.arange(0, len(forecasts), num_series)
    pv_forecasts = []

    # Extract data from Forecast objects
    # Chronos Pipeline returns numpy-based Forecast objects
    for idx in target_indices:
        f = forecasts[idx]
        pv_forecasts.append(f)

    return pv_forecasts

# --- 3. Metric Calculation Helper (User Provided) ---
def calculate_item_mase(y_true, y_pred, y_history, seasonality=96):
    """Calculates MASE for a single item."""
    forecast_mae = np.mean(np.abs(y_true - y_pred))
    if len(y_history) <= seasonality:
        return np.inf
    naive_errors = np.abs(y_history[seasonality:] - y_history[:-seasonality])
    naive_mae = np.mean(naive_errors)
    if naive_mae == 0:
        return np.inf
    return forecast_mae / naive_mae

def calculate_item_mape(y_true, y_pred, epsilon=1e-10):
    """Calculates MAPE."""
    mask = y_true > epsilon
    if np.sum(mask) == 0:
        return np.nan
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return mape

def calculate_item_wmape(y_true, y_pred):
    """Calculates wMAPE."""
    total_abs_error = np.sum(np.abs(y_true - y_pred))
    total_actuals = np.sum(np.abs(y_true))
    if total_actuals == 0:
        return np.inf
    return (total_abs_error / total_actuals) * 100

# --- 4. Main Plotting Function (User Provided) ---
def plot_model_comparison(train_df, test_df, model_predictions,
                          plot_history_length=200, prediction_length=96, seasonality=96):

    # A. Setup Data
    full_data = pd.concat([train_df, test_df]).sort_values(['item_id', 'timestamp'])
    item_ids = sorted(full_data['item_id'].unique())
    num_plots = len(item_ids)

    if num_plots > 20:
        print(f"Warning: Plotting {num_plots} series will create a very large image.")

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    fig, axs = plt.subplots(num_plots, 1, figsize=(15, 6 * num_plots), sharex=False)
    if num_plots == 1: axs = [axs]

    print(f"Generating plots for {num_plots} items...")

    for i, item_id in enumerate(item_ids):
        ax = axs[i]

        item_full_data = full_data[full_data['item_id'] == item_id].set_index('timestamp')
        item_test_data = test_df[test_df['item_id'] == item_id].set_index('timestamp')
        item_train_data = train_df[train_df['item_id'] == item_id].set_index('timestamp')

        # 1. History Context
        history_context = item_full_data.iloc[-(plot_history_length + prediction_length):]

        # 2. Ground Truth
        # For our visualization flow, we might need to ensure alignment.
        # Ideally ground_truth_future matches the prediction timestamps.
        # We will assume model_predictions aligns with the END of the dataset for this demo.

        # Check if we have predictions for this item
        has_preds = False
        for _, pred_df_all in model_predictions.items():
            if not pred_df_all[pred_df_all['item_id'] == item_id].empty:
                has_preds = True
                break

        if not has_preds: continue

        # --- D. Plot Background ---
        ax.plot(history_context.index, history_context['pv_value'],
                label='Actual Ground Truth', color='black', linewidth=2, alpha=0.6)

        # --- E. Plot Each Model ---
        for idx, (model_name, pred_df_all) in enumerate(model_predictions.items()):
            item_preds = pred_df_all[pred_df_all['item_id'] == item_id].set_index('timestamp')
            if item_preds.empty: continue

            # Align Ground Truth to Prediction
            start_date = item_preds.index[0]
            end_date = item_preds.index[-1]
            ground_truth_future = item_test_data.loc[start_date:end_date]['pv_value']

            # Highlight Forecast Window (only once)
            if idx == 0 and len(ground_truth_future) > 0:
                 ax.axvspan(start_date, end_date, color='gray', alpha=0.1, label="Forecast Window")
                 ax.axvline(x=start_date, color='black', linestyle=':', linewidth=1)

            # 1. Calculate Metrics
            try:
                # Align values
                y_pred = item_preds['predictions'].values
                y_true = ground_truth_future.values

                # Truncate to min length
                min_len = min(len(y_pred), len(y_true))
                y_pred = y_pred[:min_len]
                y_true = y_true[:min_len]

                history_for_metric = item_train_data['pv_value'].values

                mase_score = calculate_item_mase(y_true, y_pred, history_for_metric, seasonality)
                mape_score = calculate_item_mape(y_true, y_pred)
                wmape = calculate_item_wmape(y_true, y_pred)

                metrics_label = f"MASE: {mase_score:.2f} | MAPE: {mape_score:.1f}% | wMAPE: {wmape: .1f}"
            except Exception as e:
                metrics_label = f"MASE: N/A ({str(e)})"

            # 2. Plot Predictions
            color = colors[idx % len(colors)]
            label_text = f'{model_name} | {metrics_label}'

            ax.plot(item_preds.index, y_pred,
                    label=label_text, color=color, linewidth=2, linestyle='--')

            if '0.1' in item_preds.columns and '0.9' in item_preds.columns:
                ax.fill_between(item_preds.index, item_preds['0.1'], item_preds['0.9'], color=color, alpha=0.15)

        ax.set_title(f"Item {item_id}: Forecast Comparison", fontsize=14, fontweight='bold')
        ax.set_ylabel("PV Value")
        ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(CHECKPOINT_DIR, "forecast_plot.png")
    plt.savefig(plot_path)
    print(f"Plot saved to {plot_path}")
    # plt.show()

def main():
    # 1. Load Data
    print(f"Loading data from {DATA_PATH}...")
    df = pd.read_parquet(DATA_PATH)
    if "series_id" in df.columns:
        df = df.rename(columns={"series_id": "item_id", "pv": "pv_value"})

    if 'time' in df.columns:
        df['timestamp'] = pd.to_datetime(df['time'])
        if df['timestamp'].dt.tz is not None:
             df['timestamp'] = df['timestamp'].dt.tz_localize(None)

    df = df.sort_values(['item_id', 'timestamp'])
    df = df.set_index('timestamp', drop=False) # Ensure index

    # Test Dataset (using Stride to sparse sample)
    test_dataset = MultimodalDataset(
        df,
        prediction_length=PREDICTION_LENGTH,
        context_length=CONTEXT_LENGTH,
        mode="test",
        use_precomputed=True,
        stride=STRIDE
    )

    # 2. Load Model
    wrapper_model, pipeline = load_multimodal_model()

    # 3. Predict Loop
    results = []

    print("Generating predictions...")
    batch_size = 4

    # Manual loop
    for idx_batch in range(0, len(test_dataset), batch_size):
        batch_items = [test_dataset[i] for i in range(idx_batch, min(idx_batch+batch_size, len(test_dataset)))]
        #from multimodal_dataset import collate_fn
        batch = collate_fn(batch_items)

        # pv_forecasts is a list of Forecast objects
        pv_forecasts = generate_multimodal_predictions(wrapper_model, pipeline, batch)

        # Process results
        for b, forecast in enumerate(pv_forecasts):
            s_id, start_idx = test_dataset.samples[idx_batch + b]

            # Timestamps
            series_data = df[df['item_id'] == s_id]
            pred_start_idx = start_idx + CONTEXT_LENGTH
            timestamps = series_data.iloc[pred_start_idx : pred_start_idx + PREDICTION_LENGTH]['timestamp'].values

            # Extract Quantiles
            # Chronos Forecast object usually has .quantile(q)
            # Or .mean .median
            # Let's inspect what 'forecast' is. It is typically a ChronosForecast or similar.
            # Assuming it supports key access or method call.
            # Stats
            samples = forecast.samples # (Num_Samples, Pred_Len)
            # We want the median as the main prediction for metrics
            # But plotting function expects 'predictions', '0.1', '0.9'

            median = np.median(samples, axis=0)
            p10 = np.percentile(samples, 10, axis=0)
            p90 = np.percentile(samples, 90, axis=0)

            for t, med, low, high in zip(timestamps, median, p10, p90):
                results.append({
                    "item_id": s_id,
                    "timestamp": t,
                    "predictions": med, # Renamed to match plotting expects
                    "0.1": low,
                    "0.9": high
                })

    pred_df = pd.DataFrame(results)

    # 4. Visualize
    # Align format: The user's plot function expects a dictionary {ModelName: DataFrame}
    # DataFrame must have [item_id, timestamp, predictions, 0.1, 0.9]

    models_to_plot = {
        "Multimodal Chronos": pred_df
    }

    plot_model_comparison(
        train_df=df[df['timestamp'] < pred_df['timestamp'].min()],
        test_df=df, # test_df usually contains the ground truth for the future window
        model_predictions=models_to_plot,
        prediction_length=PREDICTION_LENGTH
    )

if __name__ == "__main__":
    main()


Loading data from /content/drive/MyDrive/FM_project/dataset/skippd_train_embeddings.parquet...
Preprocessing dataset into tensors for speed...
Initializing Multimodal Architecture...
Using precomputed embeddings. Skipping Vision Backbone load.
Loading Base Chronos Pipeline...
Loading LoRA Adapter from /content/multimodal_checkpoints_dim_16_batch_16/chronos_lora_adapter...
Loading Vision Projector from /content/multimodal_checkpoints_dim_16_batch_16/vision_projector.pth...
Generating predictions...


TypeError: Unexpected keyword arguments: ['group_ids', 'num_samples'].

In [18]:
!zip -r /content/multimodal_checkpoints_dim_8_batch_32.zip /content/multimodal_checkpoints

  adding: content/multimodal_checkpoints/ (stored 0%)
  adding: content/multimodal_checkpoints/train_log_dim_8.txt (deflated 75%)
  adding: content/multimodal_checkpoints/chronos_lora_adapter/ (stored 0%)
  adding: content/multimodal_checkpoints/chronos_lora_adapter/adapter_config.json (deflated 59%)
  adding: content/multimodal_checkpoints/chronos_lora_adapter/README.md (deflated 66%)
  adding: content/multimodal_checkpoints/chronos_lora_adapter/adapter_model.safetensors (deflated 7%)
  adding: content/multimodal_checkpoints/vision_projector.pth (deflated 23%)
